In [1]:
import sys, os
try:
    _d = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.exists(os.path.join(_d, '_config.yml')): break
    _d = os.path.dirname(_d)
if _d not in sys.path: sys.path.insert(0, _d)
from ai_widget_loader import load_ai_widget
load_ai_widget()

# עריכה וחישובים בסיסיים בטבלה

אחרי שלמדנו לטעון טבלה ולבחור מתוכה נתונים, נשתמש ב־Pandas כדי לחשב עמודות חדשות ולענות על שאלות כימיות פשוטות. הדגש כאן הוא לא על תחביר מתוחכם, אלא על רצף עבודה ברור: מתחילים מטבלה, מוסיפים חישוב, ואז מפרשים את התוצאה.


```{note}
אפשר להוריד את קובץ הנתונים לעבודה עצמאית: {download}`compound_phase_points.csv <files/compound_phase_points.csv>`.
```


In [2]:
import pandas as pd

pd.options.display.max_rows = 12
pd.options.display.max_columns = 10

df = pd.read_csv("files/compound_phase_points.csv")
display(df)


,compound,formula,family,molar_mass_g_mol,melting_point_C,boiling_point_C,polarity
0,water,H2O,inorganic,18.02,0.0,100.0,polar
1,methanol,CH4O,alcohol,32.04,-97.6,64.7,polar
2,ethanol,C2H6O,alcohol,46.07,-114.1,78.4,polar
3,1-propanol,C3H8O,alcohol,60.10,-126.0,97.2,polar
4,acetone,C3H6O,ketone,58.08,-94.7,56.1,polar
...,...,...,...,...,...,...,...
11,chloroform,CHCl3,halogenated,119.38,-63.5,61.2,weakly polar
12,ethane,C2H6,alkane,30.07,-182.8,-88.6,nonpolar
13,butane,C4H10,alkane,58.12,-138.3,-0.5,nonpolar
14,naphthalene,C10H8,aromatic,128.17,80.2,218.0,nonpolar


## יצירת עמודה מחושבת

נקודות התכה ורתיחה נתונות כאן במעלות צלזיוס. אם נרצה לעבוד בקלווין, נוכל ליצור עמודות חדשות. הפעולה מתבצעת על כל הערכים בעמודה.


In [3]:
df["melting_point_K"] = df["melting_point_C"] + 273.15
df["boiling_point_K"] = df["boiling_point_C"] + 273.15

display(df[["compound", "melting_point_K", "boiling_point_K"]])


,compound,melting_point_K,boiling_point_K
0,water,273.15,373.15
1,methanol,175.55,337.85
2,ethanol,159.05,351.55
3,1-propanol,147.15,370.35
4,acetone,178.45,329.25
...,...,...,...
11,chloroform,209.65,334.35
12,ethane,90.35,184.55
13,butane,134.85,272.65
14,naphthalene,353.35,491.15


נחשב גם את טווח הטמפרטורות שבו החומר נמצא במצב נוזלי בלחץ אטמוספרי, בקירוב: נקודת הרתיחה פחות נקודת ההתכה.


In [4]:
df["liquid_range_C"] = df["boiling_point_C"] - df["melting_point_C"]

display(df[["compound", "melting_point_C", "boiling_point_C", "liquid_range_C"]])


,compound,melting_point_C,boiling_point_C,liquid_range_C
0,water,0.0,100.0,100.0
1,methanol,-97.6,64.7,162.3
2,ethanol,-114.1,78.4,192.5
3,1-propanol,-126.0,97.2,223.2
4,acetone,-94.7,56.1,150.8
...,...,...,...,...
11,chloroform,-63.5,61.2,124.7
12,ethane,-182.8,-88.6,94.2
13,butane,-138.3,-0.5,137.8
14,naphthalene,80.2,218.0,137.8


כעת הטבלה לא רק מאחסנת נתונים מקוריים, אלא גם תוצאה שחישבנו מהם. זה אחד השימושים החשובים של Pandas בניתוח נתונים מדעיים.


## מיון לפי ערך

כדי לראות אילו חומרים רותחים בטמפרטורות נמוכות או גבוהות, נמיין את הטבלה לפי נקודת הרתיחה.


In [5]:
boiling_sorted = df.sort_values("boiling_point_C")
display(boiling_sorted[["compound", "family", "boiling_point_C"]])


,compound,family,boiling_point_C
12,ethane,alkane,-88.6
13,butane,alkane,-0.5
5,diethyl ether,ether,34.6
4,acetone,ketone,56.1
11,chloroform,halogenated,61.2
...,...,...,...
0,water,inorganic,100.0
9,toluene,aromatic,110.6
10,acetic acid,carboxylic acid,118.1
14,naphthalene,aromatic,218.0


מהטבלה הממוינת אפשר לראות, למשל, ש־diethyl ether רותח בטמפרטורה נמוכה יחסית, בעוד acetic acid רותחת בטמפרטורה גבוהה יותר מרוב התרכובות בטבלה.


## סינון לפי תנאי

נשאל שאלה כימית פשוטה: אילו חומרים בטבלה צפויים להיות נוזלים בטמפרטורת החדר, נניח `25°C`? חומר יהיה נוזלי אם טמפרטורת החדר גבוהה מנקודת ההתכה ונמוכה מנקודת הרתיחה.


In [6]:
room_temperature_C = 25

is_liquid_at_room_temperature = (
    (df["melting_point_C"] < room_temperature_C)
    & (df["boiling_point_C"] > room_temperature_C)
)

liquids = df.loc[is_liquid_at_room_temperature]
display(liquids[["compound", "melting_point_C", "boiling_point_C"]])


,compound,melting_point_C,boiling_point_C
0,water,0.0,100.0
1,methanol,-97.6,64.7
2,ethanol,-114.1,78.4
3,1-propanol,-126.0,97.2
4,acetone,-94.7,56.1
5,diethyl ether,-116.3,34.6
6,hexane,-95.0,68.7
7,heptane,-90.6,98.4
8,benzene,5.5,80.1
9,toluene,-95.0,110.6


המשתנה `is_liquid_at_room_temperature` הוא מסיכה בוליאנית: עבור כל שורה הוא מכיל `True` או `False`. כאשר מעבירים את המסיכה ל־`loc`, Pandas מחזירה רק את השורות שבהן התנאי נכון.


## סינון לפי טקסט

אפשר לסנן גם לפי ערכים טקסטואליים. לדוגמה, נציג רק אלכוהולים:


In [7]:
alcohols = df.loc[df["family"] == "alcohol"]
display(alcohols[["compound", "formula", "boiling_point_C"]])


,compound,formula,boiling_point_C
1,methanol,CH4O,64.7
2,ethanol,C2H6O,78.4
3,1-propanol,C3H8O,97.2


כאן התנאי `df["family"] == "alcohol"` בודק כל שורה בנפרד. זו אותה דרך חשיבה כמו ב־NumPy: פעולה אחת יוצרת מערך או סדרה של ערכי אמת/שקר, ואז משתמשים בה כדי לבחור נתונים.


## הסרת עמודות מתצוגה

לפעמים נרצה להציג טבלה מצומצמת יותר. אפשר לבחור את העמודות הרצויות במקום למחוק עמודות מהטבלה המקורית:


In [8]:
summary_columns = [
    "compound",
    "family",
    "melting_point_C",
    "boiling_point_C",
    "liquid_range_C",
]

summary = df[summary_columns]
display(summary)


,compound,family,melting_point_C,boiling_point_C,liquid_range_C
0,water,inorganic,0.0,100.0,100.0
1,methanol,alcohol,-97.6,64.7,162.3
2,ethanol,alcohol,-114.1,78.4,192.5
3,1-propanol,alcohol,-126.0,97.2,223.2
4,acetone,ketone,-94.7,56.1,150.8
...,...,...,...,...,...
11,chloroform,halogenated,-63.5,61.2,124.7
12,ethane,alkane,-182.8,-88.6,94.2
13,butane,alkane,-138.3,-0.5,137.8
14,naphthalene,aromatic,80.2,218.0,137.8


בשלב ראשון עדיף לעבוד כך ולא להשתמש ב־`inplace=True`: אנחנו יוצרים משתנה חדש לתצוגה או לניתוח, והטבלה המקורית נשארת זמינה אם נרצה לחזור אליה.


## תרגול קצר

1. צרו עמודה חדשה בשם `boiling_minus_room_C`, שמחשבת בכמה מעלות נקודת הרתיחה גבוהה מ־`25°C`.
2. מצאו את התרכובות שנקודת ההתכה שלהן גבוהה מ־`0°C`.
3. מיינו את הטבלה לפי `molar_mass_g_mol`. האם נקודת הרתיחה תמיד עולה עם המסה המולרית? הסתכלו על התוצאה ונסחו הסתייגות קצרה.


In [9]:
# Write your code here
